# 04 · Back-test and model package
Answers the brief's question: *which 4,000 customers would the model have contacted on 30 June, and how many actually converted?* Every customer is scored **out of fold** (by a model that never trained on them). Saves the back-test list (**artifact 8**) and the final model package (**final output 3**).

In [ ]:
import json

import joblib
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_predict

import pipeline_config as cfg

df = pd.read_csv(cfg.CLEAN_FILE)
features = cfg.feature_columns(df)
X, y = df[features], df[cfg.TARGET]

## 1. Out-of-fold propensity scores

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=cfg.RANDOM_STATE)
oof = cross_val_predict(cfg.build_gradient_boosting(df), X, y, cv=cv, method="predict_proba")[:, 1]
scored = df.assign(propensity=oof)
scored["propensity"].describe().round(3)

## 2. Eligible customers and the 4,000-customer list

In [ ]:
eligible = scored[(scored["marketing_opt_in"] == 1) & (scored["complaints_12m"] <= cfg.MAX_COMPLAINTS)]
base_rate = eligible[cfg.TARGET].mean()
print(f"eligible: {len(eligible):,} of {len(df):,} customers | eligible conversion rate {base_rate:.2%}")

target_list = (eligible.nlargest(cfg.CAMPAIGN_SIZE, "propensity")
               .assign(rank=lambda d: range(1, len(d) + 1))
               [["rank", "customer_id", "propensity", "customer_segment", "primary_product", "num_products",
                 "tenure_months", "region", "marketing_opt_in", "complaints_12m", cfg.TARGET]]
               .rename(columns={cfg.TARGET: "actual_converted"}))
target_list.to_csv(cfg.BACKTEST_FILE, index=False)
print(f"saved {len(target_list):,} customers -> {cfg.BACKTEST_FILE.name}")
target_list.head()

## 3. Back-test result

In [ ]:
expected = target_list["propensity"].sum()
actual = int(target_list["actual_converted"].sum())
random_conversions = cfg.CAMPAIGN_SIZE * base_rate
cost = cfg.CAMPAIGN_SIZE * cfg.COST_PER_CONTACT

backtest = {
    "customers_contacted": cfg.CAMPAIGN_SIZE,
    "expected_conversions": round(float(expected)),
    "actual_conversions": actual,
    "actual_conversion_rate": round(actual / cfg.CAMPAIGN_SIZE, 3),
    "eligible_base_rate": round(float(base_rate), 4),
    "lift_vs_eligible": round(actual / cfg.CAMPAIGN_SIZE / base_rate, 2),
    "random_conversions": round(float(random_conversions)),
    "profit_model_usd": int(actual * cfg.MARGIN_PER_CONVERSION - cost),
    "profit_random_usd": int(random_conversions * cfg.MARGIN_PER_CONVERSION - cost),
}
backtest["passes_backtest_lift"] = bool(backtest["lift_vs_eligible"] >= cfg.ACCEPTANCE["backtest_lift"])
pd.Series(backtest)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
labels = ["Random 4,000", "Model 4,000"]
panels = [("Conversions (actual Jul-Sep)", [backtest["random_conversions"], backtest["actual_conversions"]], "{:,.0f}"),
          ("Profit ($k)", [backtest["profit_random_usd"] / 1000, backtest["profit_model_usd"] / 1000], "${:,.1f}k")]
for ax, (title, values, fmt) in zip(axes, panels):
    ax.bar(labels, values, color=["#9aa5b1", "#2f6db5"])
    for i, v in enumerate(values):
        ax.text(i, v, fmt.format(v), ha="center", va="bottom")
    ax.set_title(title)
    ax.set_ylim(0, max(values) * 1.15)
fig.suptitle("Back-test: same budget, model-selected vs random customers")
fig.tight_layout()
fig.savefig(cfg.CHART_DIR / "09_backtest.png", dpi=150)
plt.show()

## 4. Package the final model (trained on all 20,000 customers)

In [ ]:
cfg.PACKAGE_DIR.mkdir(exist_ok=True)
final_model = cfg.build_gradient_boosting(df).fit(X, y)
joblib.dump(final_model, cfg.PACKAGE_DIR / "model.joblib")

holdout = pd.read_csv(cfg.COMPARISON_FILE, index_col="model")
metrics = {
    "holdout_gradient_boosting": holdout.loc["gradient_boosting", ["roc_auc", "pr_auc", "brier"]].to_dict(),
    "holdout_baseline_pr_auc": float(holdout.loc["baseline_logreg", "pr_auc"]),
    "backtest": backtest,
    "features": features,
    "excluded_columns": cfg.EXCLUDED,
}
(cfg.PACKAGE_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2))
print("saved model.joblib and metrics.json")

In [ ]:
h = metrics["holdout_gradient_boosting"]
readme = f"""# Cross-sell propensity model

Predicts the probability that an existing customer opens an additional product within 90 days.
Trained on the 30-Jun-2026 snapshot (20,000 customers, outcome Jul-Sep 2026).

## How to score a new snapshot (e.g. 30-Sep-2026)
1. Run `2_Code/01_data_preprocessing.ipynb` on the new extract (same cleaning rules).
2. `model = joblib.load("model.joblib")`
3. `propensity = model.predict_proba(cleaned[features])[:, 1]` (feature list in `metrics.json`).
4. Keep opted-in customers with fewer than 2 complaints; contact the top {cfg.CAMPAIGN_SIZE:,}.

## Excluded columns
- `cross_sell_product_type`, `welcome_pack_sent`: populated after the purchase (leakage)
- `gender`: protected attribute (fair-treatment policy)
- IDs and dates

## Performance
| Check | Result |
|---|---|
| Hold-out ROC-AUC | {h["roc_auc"]:.3f} |
| Hold-out PR-AUC | {h["pr_auc"]:.3f} (baseline {metrics["holdout_baseline_pr_auc"]:.3f}) |
| Back-test conversions (4,000 contacted) | {backtest["actual_conversions"]:,} vs {backtest["random_conversions"]:,} random |
| Back-test profit | ${backtest["profit_model_usd"]:,} vs ${backtest["profit_random_usd"]:,} random |

## Limitations
- One snapshot and one outcome window: re-validate every quarter before use.
- Probabilities assume a campaign mix similar to Jul-Sep 2026.
"""
(cfg.PACKAGE_DIR / "README.md").write_text(readme, encoding="utf-8")
print(readme)